# 03 · three ways to decide, in code

**how to use this:** press ▶ on each cell, top to bottom. anything in a box on the right is yours to change. change it, press ▶ again, see what happens. you can't break anything. if it gets weird: *Runtime → Restart session* and start from the top.

yesterday you **were** a decision tree, nearest neighbours and k-means. today the computer does it, on real data, in a few lines each.

In [ ]:
#@title ▶ run me first (loads the data and the tools)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import make_moons, make_blobs
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")

sns.set_theme(style="white")
ORANGE_PURPLE = ListedColormap(["#ffd8b1", "#f7a8f7"])

titanic = sns.load_dataset("titanic")
data = pd.DataFrame({
    "class": titanic.pclass,
    "is_female": (titanic.sex == "female").astype(int),
    "age": titanic.age.fillna(titanic.age.median()),   # fill the 177 missing ages with the middle age
    "fare": titanic.fare,
})
survived = titanic.survived
X_train, X_test, y_train, y_test = train_test_split(data, survived, test_size=0.2, random_state=42)

def regions(model, X, y, title):
    """colour the whole plane by what the model would say, then draw the dots on top"""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 300), np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 300))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(6, 6))
    plt.contourf(xx, yy, zz, cmap=ORANGE_PURPLE, alpha=0.9)
    plt.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "#ff6a00", "#8a00c2"), s=18)
    plt.xticks([]); plt.yticks([]); plt.title(title)
    plt.show()

print(f"ready ✅  {len(X_train)} passengers to learn from, {len(X_test)} kept hidden for the exam")

## the lazy guess
first, the number to beat: guess *didn't survive* for everyone.

In [ ]:
#@title how good is the lazy guess?
lazy = (y_test == 0).mean()
print(f"guess 'didn't survive' for every hidden passenger: right {lazy:.0%} of the time")

## 1 · decision tree
the computer picks the questions itself. you only choose **how many questions deep** it's allowed to go.

In [ ]:
#@title grow a tree
depth = 2 #@param {type:"slider", min:1, max:10, step:1}
tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
print(f"right on the passengers it learned from: {tree.score(X_train, y_train):.0%}")
print(f"right on the hidden passengers (the exam): {tree.score(X_test, y_test):.0%}")
plt.figure(figsize=(18, 7))
plot_tree(tree, feature_names=list(data.columns), class_names=["didn't survive", "survived"], filled=True, rounded=True, max_depth=3, fontsize=14 if depth <= 2 else 10, impurity=False, proportion=True)
plt.title("the tree (first 3 levels)" if depth > 3 else "the tree")
plt.show()

how to read it: `is_female <= 0.5` means *is this a man?* left = yes, right = no.

at depth 2 it asks about sex first. then **class** for the women, and **age** for the men: boys under 6½ mostly survived. *women and children first*, and nobody told it.

In [ ]:
#@title when does asking more questions stop helping?
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train)); test_acc.append(t.score(X_test, y_test))
plt.figure(figsize=(8, 4))
plt.plot(depths, train_acc, "o-", color="#c7c7c7", label="passengers it learned from")
plt.plot(depths, test_acc, "o-", color="black", label="hidden passengers (the exam)")
plt.xlabel("how many questions deep"); plt.ylabel("right"); plt.legend(); plt.title("overfitting, in one picture")
plt.show()

the grey line keeps climbing: it's **memorising** (that's laina). the black line stops improving. the gap between them is overfitting.

In [ ]:
#@title would YOU have survived?
passenger_class = 3 #@param [1, 2, 3]
sex = "male" #@param ["female", "male"]
age = 25 #@param {type:"slider", min:1, max:80, step:1}
fare = 8 #@param {type:"number"}

you = pd.DataFrame([{"class": passenger_class, "is_female": int(sex == "female"), "age": age, "fare": fare}])
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
chance = tree3.predict_proba(you)[0][1]
print(f"the tree says: {'survived 🛟' if chance >= 0.5 else 'did not survive 🌊'}  ({chance:.0%} of passengers like you survived)")

## 2 · nearest neighbours
the ml-playground dots, but in code. you choose **k** and how messy the dots are.

In [ ]:
#@title nearest neighbours on two moons
k = 3 #@param {type:"slider", min:1, max:50, step:1}
messiness = 0.3 #@param {type:"slider", min:0, max:0.6, step:0.05}
X, y = make_moons(n_samples=200, noise=messiness, random_state=1)
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=1)
knn = KNeighborsClassifier(n_neighbors=k).fit(Xa, ya)
print(f"k = {k}: right on the hidden dots {knn.score(Xb, yb):.0%}")
regions(knn, X, y, f"nearest neighbours, k = {k}")

try **k = 1** (little islands around every odd dot) and **k = 50** (too blurry). somewhere in the middle is best.

In [ ]:
#@title nearest neighbours on the titanic
k = 5 #@param {type:"slider", min:1, max:50, step:1}
knn_t = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)   # scale first: fare in £ would drown age in years
print(f"k = {k}: right on the hidden passengers {knn_t.score(X_test, y_test):.0%}")

## 3 · clustering
no labels at all. you only say **how many groups** to look for.

In [ ]:
#@title k-means on blobs
how_many_blobs = 4 #@param {type:"slider", min:2, max:8, step:1}
k = 4 #@param {type:"slider", min:1, max:8, step:1}
method = "k-means" #@param ["k-means", "agglomerative"]
X, _ = make_blobs(n_samples=300, centers=how_many_blobs, cluster_std=1.1, random_state=7)
model = KMeans(n_clusters=k, n_init=10, random_state=0) if method == "k-means" else AgglomerativeClustering(n_clusters=k)
groups = model.fit_predict(X)
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=groups, cmap="tab10", s=18)
if method == "k-means":
    plt.scatter(*model.cluster_centers_.T, marker="s", s=200, facecolors="white", edgecolors="black", linewidths=2)
plt.xticks([]); plt.yticks([]); plt.title(f"{method} looking for {k} groups")
plt.show()

set `k` lower or higher than the real number of blobs. k-means will happily find groups that aren't there.

In [ ]:
#@title the smiley face, again
k = 3 #@param {type:"slider", min:2, max:6, step:1}
rng = np.random.default_rng(0)
t = rng.uniform(0, 2 * np.pi, 400); face = np.c_[np.cos(t), np.sin(t)] * 3
eyes = np.r_[rng.normal([-1, 1], .15, (60, 2)), rng.normal([1, 1], .15, (60, 2))]
s = rng.uniform(np.pi * 1.15, np.pi * 1.85, 120); mouth = np.c_[np.cos(s), np.sin(s) + .4] * 1.6
smiley = np.r_[face, eyes, mouth]
groups = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(smiley)
plt.figure(figsize=(6, 6)); plt.scatter(*smiley.T, c=groups, cmap="tab10", s=12); plt.axis("equal"); plt.axis("off")
plt.title("k-means sees blobs, not a face"); plt.show()

## 4 · regression: predicting a number
not *which group*, but *how much*. here: how far a car goes on a litre of petrol, from its horsepower.

In [ ]:
#@title draw the line
mpg = sns.load_dataset("mpg").dropna()
X_hp = mpg[["horsepower"]]
km_per_litre = mpg["mpg"] * 0.425
line = LinearRegression().fit(X_hp, km_per_litre)
plt.figure(figsize=(8, 4))
plt.scatter(mpg.horsepower, km_per_litre, s=10, color="#c7c7c7")
hp = np.linspace(40, 230, 100)
plt.plot(hp, line.predict(pd.DataFrame({"horsepower": hp})), color="black", linewidth=2)
plt.xlabel("horsepower"); plt.ylabel("km per litre"); plt.title(f"{len(mpg)} real cars from the 1970s and 80s")
plt.show()

In [ ]:
#@title predict a car
horsepower = 100 #@param {type:"slider", min:40, max:230, step:5}
guess = line.predict(pd.DataFrame({"horsepower": [horsepower]}))[0]
print(f"a {horsepower} horsepower car: about {guess:.1f} km per litre")

## 5 · the model race
the syllabus lists six famous models. same passengers, same exam. who wins?

In [ ]:
#@title race them
models = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "nearest neighbours": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": DecisionTreeClassifier(max_depth=3, random_state=0),
    "random forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0),
    "support vector machine": make_pipeline(StandardScaler(), SVC()),
    "naive bayes": GaussianNB(),
}
scores = {name: m.fit(X_train, y_train).score(X_test, y_test) for name, m in models.items()}
scores = pd.Series(scores).sort_values()
ax = scores.mul(100).plot.barh(color="black", figsize=(8, 4), xlim=(50, 90), title="right on the hidden passengers (%)")
ax.axvline((y_test == 0).mean() * 100, color="#c7c7c7", linestyle="--"); ax.bar_label(ax.containers[0], fmt="%.0f%%")
plt.show()

the dashed line is the lazy guess. every model beats it, and they're all **within a few percent of each other**. on most real problems the data matters more than which model you pick.

- **random forest** = a few hundred decision trees voting
- **logistic regression** = a straight line that splits survived from didn't
- **support vector machine** = the widest possible gap between the two groups
- **naive bayes** = probabilities, multiplied together